# ДЗ 6 · Заняття 6. MCP-сервери: основи

**Підключаємо готовий MCP-сервер: від терміналу до агента на Python з Gemini**

Прізвище Ім'я: `______________` (перейменуйте файл на `ДЗ6_Прізвище_Імʼя` перед здачею)

---

### Як користуватися цим ноутбуком

1. Відкрити в Google Colab: **File → Upload notebook** (або відкрити з GitHub).
2. Ключ Gemini покласти в секрети Colab, а **не** в код: зліва 🔑 **Secrets → Add new secret**, ім'я `GEMINI_API_KEY`, значення — ключ курсу, увімкнути **Notebook access**.
3. Виконувати клітинки згори вниз (**Runtime → Run all** теж працює).

У ноутбуці вже збережено вивід реального прогону (20.09.2026). Після власного запуску Colab перезапише його вашим — так і має бути.

### Що варто знати про версії (перевірено на практиці)

| Що | Чому саме так |
|---|---|
| `fastmcp` останній (4.x) | команди `fastmcp list` і `fastmcp call` з'явилися лише у 4.x; у 2.x їх немає взагалі (`Unknown command "list"`) |
| `google-genai<2` | у 2.x рядок `tools=[mcp_client.session]` падає з `TypeError: cannot pickle '_asyncio.Task' object`: бібліотека робить `config.model_copy(deep=True)`, а живу MCP-сесію скопіювати неможливо. Автоматичний виклик MCP-інструментів із сесії живе у гілці 1.x |
| `gemini-3.7-flash` замість `gemini-2.5-flash` | модель із методички більше не видається новим ключам: `404 NOT_FOUND … is no longer available to new users`. Актуальну назву звіряв зі списком `client.models.list()` |

### Без ключа в коді

Ключ живе лише у секретах Colab і потрапляє у змінну середовища `GEMINI_API_KEY`. Ні в одній клітинці ключ не друкується.

⚠️ Безкоштовний тариф Gemini має ліміт **20 запитів на добу на модель**. Кроки 1 і 2 моделі не використовують — їх можна ганяти скільки завгодно. Клітинки Кроку 3 і Бонусу мають вбудовані повтори на `429`/`503`, але запускати їх десятки разів поспіль не треба: ліміт спільний.

In [ ]:
# Крок 0. Бібліотеки.
# fastmcp — останній (потрібні команди list/call), google-genai — гілка 1.x (див. таблицю вище).
%pip install -q -U fastmcp "google-genai<2"

!fastmcp version

In [ ]:
# Сумісність двох свіжих бібліотек (потрібно лише для Бонусу, але вмикаємо один раз тут).
#
# fastmcp 4 додає в схему інструмента "additionalProperties": false, а конвертер
# google-genai 1.x рекурсивно заходить у КОЖНЕ значення схеми й чекає там словник.
# На булевому значенні він падає: AttributeError: 'bool' object has no attribute 'items'.
# Три рядки нижче кажуть конвертеру: не словник — не чіпай, віддай як є.
from google.genai import _mcp_utils

_orig_filter = _mcp_utils._filter_to_supported_schema
_mcp_utils._filter_to_supported_schema = (
    lambda schema: _orig_filter(schema) if isinstance(schema, dict) else schema
)
print("Патч сумісності увімкнено")

---

# Крок 1. Меню сервера

Дивимося, що вміє сервер DeepWiki. Модель тут не потрібна — ключ не витрачається.

## 1.1. Список інструментів

In [1]:
!fastmcp list https://mcp.deepwiki.com/mcp

Tools (3)

  ask_question(repoName: str | list, question: str) -> dict
    Ask any question about a GitHub repository and get an AI-powered, 
context-grounded response.

  read_wiki_contents(repoName: str) -> dict
    View documentation about a GitHub repository.

  read_wiki_structure(repoName: str) -> dict
    Get a list of documentation topics for a GitHub repository.

## 1.2. Усі інструменти сервера і що вони роблять

| Інструмент | Що робить |
|---|---|
| `read_wiki_structure` | Повертає зміст згенерованої документації репозиторію — плаский список тем і підтем («1.2 Project Structure», «3.4 Handlers System»), без самого тексту. |
| `read_wiki_contents` | Повертає сам текст документації репозиторію — те, що ховається за темами зі змісту. |
| `ask_question` | Відповідає на довільне питання про репозиторій: сервер сам шукає потрібні шматки коду й документації і будує відповідь із посиланнями на них. |

Три інструменти — три різні «глибини»: зміст → повний текст → відповідь на конкретне питання. Усі три **лише читають**: жоден нічого не створює, не змінює і не надсилає. Саме тому такий сервер безпечно віддавати моделі з автоматичним викликом (Крок 3).

## 1.3. Повна схема

In [2]:
!fastmcp list https://mcp.deepwiki.com/mcp --input-schema

Tools (3)

  ask_question(repoName: str | list, question: str) -> dict
    Ask any question about a GitHub repository and get an AI-powered, 
context-grounded response.
    Input: {"properties": {"repoName": {"anyOf": [{"type": "string"}, {"items": 
{"type": "string"}, "type": "array"}], "description": "GitHub repository or list
of repositories (max 10) in owner/repo format."}, "question": {"description": 
"The question to ask about the repository.", "type": "string"}}, "required": 
["repoName", "question"], "type": "object"}

  read_wiki_contents(repoName: str) -> dict
    View documentation about a GitHub repository.
    Input: {"properties": {"repoName": {"description": "GitHub repository in 
owner/repo format (e.g. \"facebook/react\").", "type": "string"}}, "required": 
["repoName"], "type": "object"}

  read_wiki_structure(repoName: str) -> dict
    Get a list of documentation topics for a GitHub repository.
    Input: {"properties": {"repoName": {"description": "GitHub repository

## 1.4. Розбір схеми одного інструмента

```
Інструмент: ask_question
Що він робить: відповідає природною мовою на питання про конкретний GitHub-репозиторій,
               спираючись на його код і документацію (сервер сам робить пошук і сам
               генерує відповідь своєю моделлю — не моєю).

Параметр 1: назва — repoName, тип — anyOf: string АБО array of string, обов'язковий? так
            Що в нього передавати: репозиторій у форматі власник/назва
            ("python-telegram-bot/python-telegram-bot"). Можна передати список
            до 10 репозиторіїв — тоді сервер шукає відповідь одразу в кількох.

Параметр 2: назва — question, тип — string, обов'язковий? так
            Що в нього передавати: саме питання людською мовою, без розмітки й лапок —
            "Як обробити натискання inline-кнопки?". Мова відповіді залежить від мови питання.

Чим ця схема схожа на опис get_weather з уроку 5:
   той самий скелет — name, description, parameters з JSON Schema всередині: type: object,
   properties з типами й описами, список required. Модель так само вибирає інструмент
   за description і заповнює аргументи за описами полів; на рівні моделі різниці
   між «моєю» функцією і чужим MCP-сервером немає взагалі.

Чим відрізняється:
   1. Схему я не писав(ла) руками — вона приїхала з сервера по протоколу (tools/list).
      Її автор — власник сервера, і він може її змінити без мого відома.
   2. repoName має anyOf (рядок АБО масив рядків) — у моєму get_weather усі параметри
      були простими типами, об'єднань я не описував(ла).
   3. Жодного enum і жодного необов'язкового параметра: обидва поля в required.
      У ДЗ 5 від мене вимагали і enum, і опційний параметр — тут автор обійшовся без них.
   4. Опис англійською і дуже короткий: одне речення без «Використовуй, коли…» /
      «Не використовуй для…», які ми писали в ДЗ 5. Дисципліна опису — на совісті автора сервера.
   5. Повертає не рядок, а структуру (dict із полем result), і по дорозі туди-назад
      працює мережа: сервер віддалений, тому можливі таймаут, 429 і зміна поведінки.
```

---

# Крок 2. Виклик інструмента без моделі

Тепер роботу моделі виконую я: сам(а) обираю інструмент і заповнюю аргументи.

## 2.1. Мій репозиторій

`python-telegram-bot/python-telegram-bot` — бібліотека, на якій тримається Telegram-частина мого агента з ДЗ 4–5 (дайджест пошти з кнопками «додати подію в календар»). Репозиторій занадто великий, щоб тримати його в голові, і при цьому не той, що на занятті (`modelcontextprotocol/python-sdk`).

In [ ]:
REPO = "python-telegram-bot/python-telegram-bot"
QUESTION_CLI = "Як обробити натискання inline-кнопки через CallbackQueryHandler і відповісти на callback?"
print("Репозиторій:", REPO)

## 2.2. Структура документації репозиторію

Сервер віддає результат як JSON **одним рядком**, де всі переноси — це `\n` усередині рядка. У терміналі це ще читабельно, у Colab — суцільна кишка з горизонтальною прокруткою. Тому вивід команди пропускаємо через однорядковий Python, який дістає поле `result` і друкує його з живими переносами. Сама команда — точно та, що в методичці; змінилося лише оформлення виводу.

In [3]:
!fastmcp call https://mcp.deepwiki.com/mcp read_wiki_structure repoName={REPO} \
    | python3 -c "import sys, json; print(json.load(sys.stdin)['result'])"

Available pages for python-telegram-bot/python-telegram-bot:

- 1 Overview
  - 1.1 Installation and Setup
  - 1.2 Project Structure
  - 1.3 Contributing Guidelines
- 2 Core Bot API (telegram package)
  - 2.1 Bot and ExtBot Classes
  - 2.2 Message Object and Shortcuts
  - 2.3 Chat and User Objects
  - 2.4 Update Object and Update Types
  - 2.5 Error Handling and Exception Classes
  - 2.6 Request Layer and HTTP Client
  - 2.7 Serialization and Deserialization
  - 2.8 CallbackQuery and Inline Buttons
  - 2.9 Inline Queries and Results
- 3 Extension Framework (telegram.ext)
  - 3.1 Application Architecture
  - 3.2 ApplicationBuilder
  - 3.3 Updater and Update Fetching
  - 3.4 Handlers System
  - 3.5 Filters System
  - 3.6 ConversationHandler
  - 3.7 JobQueue and Scheduled Tasks
  - 3.8 CallbackContext and ContextTypes
  - 3.9 Persistence and State Management
  - 3.10 Error Handling in Application
- 4 Advanced Features
  - 4.1 Payments Integration
  - 4.2 Telegram Passport
  - 4.3 Rate Limi

## 2.3. Власне питання про репозиторій

Аргумент зі пробілами беремо в лапки **цілком**: `"question=..."`, а не `question="..."`. Вивід — так само через розпакування `result`.

Службові рядки (`UserWarning` про зберігання токенів і `Received INFO from server`) ідуть у stderr — Colab показує їх окремо, у відповідь вони не потрапляють. `INFO` тут означає, що сервер повідомляє про хід обробки: ask_question думає 15–40 секунд.

In [4]:
!fastmcp call https://mcp.deepwiki.com/mcp ask_question repoName={REPO} "question={QUESTION_CLI}" \
    | python3 -c "import sys, json; print(json.load(sys.stdin)['result'])"

Щоб обробити натискання inline-кнопки та відповісти на callback, вам потрібно використовувати `CallbackQueryHandler` для перехоплення `CallbackQuery` та метод `answer_callback_query` об'єкта `Bot` або `ExtBot` для надсилання відповіді.   

## Обробка натискання inline-кнопки за допомогою `CallbackQueryHandler`

`CallbackQueryHandler` використовується для обробки оновлень, які містять `CallbackQuery` . Коли користувач натискає inline-кнопку, Telegram надсилає `Update` з об'єктом `CallbackQuery` .

Ви можете створити екземпляр `CallbackQueryHandler`, передавши функцію зворотного виклику та необов'язковий `pattern` або `game_pattern` .
*   Параметр `pattern` використовується для фільтрації `callback_query.data` . Це може бути рядок, об'єкт `re.Pattern`, функція, що повертає булеве значення, або тип .
*   Параметр `game_pattern` використовується для фільтрації `callback_query.game_short_name` .

Якщо `pattern` або `game_pattern` не встановлено, `CallbackQueryHandler` оброблятиме будь-який 

## 2.4. Відповіді на питання кроку

```
Чи брала участь у цих викликах AI-модель? Чому?
   З мого боку — ні, і ключ курсу не витрачено. fastmcp call — це прямий виклик
   методу tools/call по протоколу MCP: клієнт відкрив з'єднання, передав назву
   інструмента й словник аргументів, отримав результат. Ніякої моделі в цьому
   ланцюжку немає: нема кому було читати питання і вирішувати, що викликати.

   Чесне уточнення: всередині самого сервера модель є — ask_question описаний як
   "AI-powered, context-grounded response", тобто DeepWiki генерує текст відповіді
   своєю моделлю. Але це його модель і його рахунок; для мого агента ask_question —
   звичайний інструмент, який повертає текст.

Хто в цьому кроці виконував роль моделі?
   Я. Подивив(ла)ся список інструментів, обрав(ла) потрібний під свою задачу
   (структура → read_wiki_structure, питання → ask_question), звірив(ла)ся зі схемою,
   правильно назвав(ла) параметри (repoName, не repo_name) і підставив(ла) значення.
   Саме це на Кроці 3 робитиме Gemini — за ті самі описи, які я щойно читав(ла) очима.
```

---

# Крок 3. Агент Gemini з MCP-сервером

Тепер ноутбук стає **хостом**: сам підключається до MCP-сервера через клієнт FastMCP і віддає всі його інструменти моделі. Рішення, що викликати, ухвалює Gemini; виклик виконує `google-genai` автоматично, без жодного підтвердження від людини.

## 3.0. Ключ

In [ ]:
import os

try:                                    # Colab: ключ із секретів (🔑 зліва)
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:                       # локальний запуск: ввести руками, без відлуння
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")

print("Ключ у середовищі:", bool(os.environ.get("GEMINI_API_KEY")))   # сам ключ не друкуємо

## 3.2. ПРОГНОЗ (записано ДО запуску)

```
Моє питання: "Як у репозиторії python-telegram-bot/python-telegram-bot реалізовано
              обмеження частоти запитів (rate limiting): які класи за це відповідають
              і як їх підключити до застосунку? Відповідай українською."

ПРОГНОЗ (записано до запуску):
Інструмент(и), які обере модель: ask_question
   Чому: read_wiki_structure дасть лише зміст без відповіді, read_wiki_contents —
   гору тексту, з якої ще треба вибирати. ask_question б'є точно в питання.
Аргументи, які вона передасть:
   repoName = "python-telegram-bot/python-telegram-bot"
   question = моє питання майже дослівно, українською
Скільки викликів буде: один
```

Питання підібране так, щоб без репозиторію на нього не відповісти напевно: назви класів і параметрів — це деталь конкретної бібліотеки.

## 3.3. Запуск агента

У Colab вже крутиться event loop, тому замість `asyncio.run(main())` — `await main()`.

З тієї ж причини пауза між повторами зроблена через `await asyncio.sleep()` і **поза** блоком `async with mcp_client`. Блокуючий `time.sleep()` усередині сесії зупиняє event loop: фонові задачі з'єднання з MCP-сервером не отримують керування, з'єднання рветься, і замість повтору прилітає `CancelledError`.

**Якщо сиплеться `429` або `503`** — це не помилка вашого коду:

* `429 RESOURCE_EXHAUSTED` — вичерпано добовий ліміт безкоштовного тарифу (**20 запитів на добу на модель**, спільний для ключа курсу);
* `503 UNAVAILABLE` — модель перевантажена на боці Google.

Обидва лікуються однаково: почекати або підставити в `MODEL` іншу flash-модель. Важливо, що ліміт рахується **на модель**, тож сусідня модель зазвичай ще має запас. Перевірка 21.09.2026 на ключі курсу: `gemini-3.7-flash` віддавав 429/503, а `gemini-3.5-flash` і `gemini-3.1-flash-lite` відповідали нормально.

Подивитися, що взагалі видає ваш ключ (асинхронний варіант — він працює і в Colab, і поряд з fastmcp):

```python
from google import genai

async for m in await genai.Client().aio.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)
```

In [5]:
import asyncio
from fastmcp import Client
from google import genai
from google.genai import types, errors

MODEL = "gemini-3.7-flash"      # gemini-2.5-flash з методички новим ключам більше не видається
QUESTION = (
    "Як у репозиторії python-telegram-bot/python-telegram-bot реалізовано "
    "обмеження частоти запитів (rate limiting): які класи за це відповідають "
    "і як їх підключити до застосунку? Відповідай українською."
)

mcp_client = Client("https://mcp.deepwiki.com/mcp")   # MCP-клієнт
gemini = genai.Client()                               # ключ береться зі змінної GEMINI_API_KEY


async def ask_once():
    async with mcp_client:                            # відкрили з'єднання із сервером
        config = types.GenerateContentConfig(
            temperature=0,
            tools=[mcp_client.session],               # передали моделі весь MCP-сервер
        )
        return await gemini.aio.models.generate_content(
            model=MODEL, contents=QUESTION, config=config
        )


async def main():
    # 429/503 на безкоштовному тарифі — звична справа.
    # Пауза живе ЗА МЕЖАМИ "async with" і робиться через await asyncio.sleep():
    # блокуючий time.sleep() усередині сесії зупинив би event loop, фонові задачі
    # з'єднання з MCP-сервером не отримали б керування, і замість повтору
    # прилетів би CancelledError.
    for attempt in range(4):
        try:
            response = await ask_once()
            break
        except (errors.ClientError, errors.ServerError) as e:
            if e.code in (429, 503) and attempt < 3:
                pause = 20 * (attempt + 1)
                print(f"{e.code}: чекаю {pause} с і повторюю")
                await asyncio.sleep(pause)
                continue
            raise
        except Exception as e:                   # мережа могла впасти і на підключенні до сервера
            if attempt < 3:
                pause = 20 * (attempt + 1)
                print(f"{type(e).__name__}: чекаю {pause} с і повторюю")
                await asyncio.sleep(pause)
                continue
            raise

    print(response.text)

    print("\n--- Що відбулося всередині ---")
    history = response.automatic_function_calling_history or []
    for content in history:
        for part in content.parts or []:
            if part.function_call:
                print("інструмент:", part.function_call.name)
                print("аргументи:", dict(part.function_call.args))


await main()          # у Colab/Jupyter замість asyncio.run(main())

У бібліотеці **`python-telegram-bot`** (починаючи з версії v20+) механізм обмеження частоти запитів (Rate Limiting) реалізовано на рівні модуля розширень `telegram.ext`.

Нижче наведено опис класів, які за це відповідають, а також інструкцію з їхнього підключення.

---

### 1. Ключові класи

1. **`BaseRateLimiter` (`telegram.ext.BaseRateLimiter`)**
   * Абстрактний базовий клас, що визначає інтерфейс для будь-якого механізму rate limiting.
   * Вимагає реалізації трьох основних методів:
     * `initialize()` — ініціалізація внутрішніх структур (блокувань, семафорів тощо).
     * `shutdown()` — коректне завершення роботи.
     * `process_request(...)` — основна логіка затримки/пропуску та повторних спроб виконання API-запитів.

2. **`AIORateLimiter` (`telegram.ext.AIORateLimiter`)**
   * Готова асинхронна реалізація `BaseRateLimiter`, яка працює на базі бібліотеки [`aiolimiter`](https://github.com/mjpieters/aiolimiter).
   * Забезпечує дотримання загальних лімітів Telegram API (за замов

## 3.5. Прогноз проти реальності

```
Чи збігся інструмент: так. Модель бачила всі три інструменти DeepWiki й обрала
   саме ask_question; read_wiki_structure і read_wiki_contents не чіпала жодного разу.

Чи збіглися аргументи: частково. repoName — точно як у прогнозі. А от question
   модель переписала англійською і сама підказала собі відповідь у питанні:
   "Which classes are responsible for rate limiting, and how do you configure /
   attach them to the Application / Bot?".

Чи збіглася кількість викликів: ні. Викликів було ДВА, а не один:
   1) загальне питання про реалізацію rate limiting і підключення;
   2) уточнення "What are the parameters of AIORateLimiter and how to install
      its dependencies?" — тобто модель побачила, що перша відповідь не покриває
      частину "як підключити", і доклала другий виклик.

Що модель зробила інакше, ніж я очікував(ла), і чому, на мою думку:
   По-перше, мова аргументів. Я думав(ла), що "Відповідай українською" в питанні
   означає українську скрізь. Насправді ця вимога стосується лише фінальної
   відповіді мені; з інструментом модель спілкується так, як їй зручніше, а
   репозиторій, його код і документація — англійські, тому й питання пішло
   англійською. Висновок на майбутнє: мова інтерфейсу і мова аргументів — різні речі.

   По-друге, кількість викликів. Моє питання насправді було подвійним ("які класи"
   + "як підключити"), і модель розклала його на два звернення до сервера замість
   того, щоб вигадувати недостаючу частину. Це і є та сама «гнучкість по кроку»,
   про яку йшлося в ДЗ 4: кількість кроків вирішується під час виконання, а не
   зашита наперед. Для гаманця це означає, що один мій запит легко перетворюється
   на N платних викликів — з DeepWiki це безпечно (він безкоштовний і лише читає),
   з іншим сервером я б це рахував(ла) окремо.
```

## 3.6. Контрольний прогін на іншій моделі (бонус до спостереження)

Наступного дня `gemini-3.7-flash` уперся в ліміти, і я запустив(ла) той самий код, змінивши один рядок — `MODEL = "gemini-3.5-flash"`. Питання, сервер і температура ті самі.

```
--- Що відбулося всередині ---
інструмент: ask_question
аргументи: {'repoName': 'python-telegram-bot/python-telegram-bot',
            'question': 'How is rate limiting implemented in python-telegram-bot?
                         Which classes are responsible for it, and how can they be
                         integrated/connected to an application?'}
```

Той самий інструмент, той самий репозиторій, питання знову англійською — але **один виклик замість двох**: 3.5-flash одразу склав(ла) питання так, щоб покрити обидві частини («які класи» + «як підключити»), і другого звернення не знадобилось. Відповідь вийшла не гіршою: класи `ExtBot`, `BaseRateLimiter`, `AIORateLimiter` і покроковий опис, як запит проходить через `_do_post`.

Висновок, якого не було в моєму прогнозі: **кількість викликів — це властивість не задачі, а моделі**. Той самий промпт і той самий сервер дають різну кількість звернень до інструмента залежно від того, хто саме планує. Для мого агента з ДЗ 4 це прямий аргумент на користь жорсткого ліміту на кількість викликів у циклі: інакше рахунок залежить від того, яку модель я підставив(ла) сьогодні.

---

# Крок 4. Готові сервери для мого агента

**Мої 5 інструментів з ДЗ 4–5** (агент «ранковий дайджест пошти в Telegram»): `list_new_emails`, `fetch_email_body`, `send_telegram_message`, `save_run_state`, `create_calendar_event`.

**Де шукав(ла)** (у такому порядку для кожного): каталог Connectors у Claude → офіційні сайти власників сервісу (Google Workspace, Telegram) → офіційний реєстр `registry.modelcontextprotocol.io` → еталонні сервери `github.com/modelcontextprotocol/servers` → пошук по GitHub.

---

### Інструмент 1: `list_new_emails` — нові листи зі скриньки (IMAP, за курсором UID)

```
Готовий MCP-сервер: знайдено частково
Посилання: офіційний Gmail MCP — https://gmailmcp.googleapis.com/mcp/v1
           (інструкція: https://developers.google.com/workspace/guides/configure-mcp-servers)
           чистий IMAP від спільноти — https://github.com/mattias242/mcp-imap-server
                                       https://github.com/rsilvestre/email-mcp
Автор: Google (офіційна компанія, статус Developer Preview) / решта — спільнота
Тип сервера: віддалений (Gmail MCP) / локальний (IMAP-сервери спільноти)
Які інструменти сервера мені потрібні: пошук і перелік листів + get_message
Чи є серед них дії на запис, видалення, надсилання: ТАК — Gmail MCP уміє create_draft,
   forward, create_filter, create_label, delete_label, накидати мітки. Тобто разом
   із читанням я отримую повний набір «писати від мого імені».
Які права/доступи потрібні: OAuth до Gmail (read+write одним пакетом) або логін
   і пароль застосунку для IMAP-серверів спільноти
Мій висновок: напишу власний інструмент. Три причини:
   1) агент ходить у КІЛЬКА скриньок, і не всі вони Gmail — Gmail MCP покриває одну;
   2) мені потрібен курсор UID + UIDVALIDITY між запусками, і жоден сервер його не
      віддає; без нього «нові листи» щоранку визначити неможливо, а дайджест
      перетвориться на повтор учорашнього;
   3) за архітектурою з ДЗ 4 модель, яка бачить вміст листів, не має ЖОДНОГО
      інструмента (ізоляція недовіреного вмісту). Сервер із write-інструментами
      поруч із текстом чужих листів — це прямий шлях до prompt injection.
   Мій варіант на imaplib — близько 40 рядків і повний контроль курсора.
```

---

### Інструмент 2: `fetch_email_body` — тіло конкретного листа за UID

```
Готовий MCP-сервер: знайдено частково (ті самі, що в п.1: Gmail MCP — get_message,
   IMAP-сервери спільноти — fetch/read_email)
Посилання: https://developers.google.com/workspace/guides/configure-mcp-servers
           https://github.com/rsilvestre/email-mcp
Автор: Google / спільнота
Тип сервера: віддалений / локальний
Які інструменти сервера мені потрібні: get_message (повний вміст листа за id)
Чи є серед них дії на запис, видалення, надсилання: так — ті самі, що в п.1;
   окремого «тільки читати» режиму сервер не дає
Які права/доступи потрібні: ті самі OAuth-скоупи Gmail
Мій висновок: напишу власний інструмент — той самий з'єднувач IMAP, що й у п.1.
   Додатковий аргумент: мені треба самому обирати частину text/plain і обрізати
   тіло до ліміту символів, інакше пачка з 50 листів рознесе контекст і рахунок.
   Готовий сервер віддає лист «як є» і такої ручки не має.
```

---

### Інструмент 3: `send_telegram_message` — надіслати дайджест власнику

```
Готовий MCP-сервер: знайдено, але лише від спільноти
Посилання: https://github.com/AdeshAtole/telegram-notifier-mcp
           https://github.com/siavashdelkhosh81/telegram-bot-mcp-server
           https://github.com/chigwell/telegram-mcp  (Telethon, працює від імені
           ОСОБИСТОГО акаунта, а не бота)
Автор: спільнота. Офіційного сервера від Telegram немає — у каталозі Connectors,
   у registry.modelcontextprotocol.io і серед еталонних серверів його теж немає
Тип сервера: локальний (npx / uvx поруч з агентом)
Які інструменти сервера мені потрібні: send_message — і цього мало: мені потрібні
   inline-клавіатура (reply_markup), повернення message_id надісланого повідомлення
   і edit_message_reply_markup, щоб гасити кнопки після натискання. Жоден зі
   знайдених серверів не дає всі три
Чи є серед них дії на запис, видалення, надсилання: ТАК, це суто write-інструмент;
   у Telethon-варіанті ще й керування чатами та видалення повідомлень
Які права/доступи потрібні: токен бота; у Telethon-варіанті — сесія особистого
   акаунта, тобто доступ до всього мого листування
Мій висновок: напишу власний інструмент. Надсилання повідомлення — це один POST
   на api.telegram.org; чужий сервер додав би залежність, ще один процес і ризик
   заради нуля користі, та ще й не закрив би саме ту частину, яка мені потрібна
   (кнопки і їх редагування).
```

---

### Інструмент 4: `save_run_state` — курсори UID, дата дайджесту, pending_events

```
Готовий MCP-сервер: не знайдено — і не потрібен
Посилання: шукав(ла) у registry.modelcontextprotocol.io і серед еталонних серверів
   github.com/modelcontextprotocol/servers; найближче за змістом — reference-сервер
   Filesystem, але він дає доступ до файлів узагалі, а не до мого стану
Автор: —
Тип сервера: —
Які інструменти сервера мені потрібні: —
Чи є серед них дії на запис, видалення, надсилання: Filesystem — так, запис і
   видалення довільних файлів
Які права/доступи потрібні: доступ до файлової системи там, де крутиться агент
Мій висновок: власний інструмент. Стан — внутрішня річ раннера: json.dump у файл
   під блокуванням. MCP тут означав би загорнути два рядки коду в мережевий
   протокол і додати нову точку збою. Це той випадок, коли шукати сервер узагалі
   не варто було — і це теж нормальна відповідь.
```

---

### Інструмент 5: `create_calendar_event` — подія в Google Calendar після натискання кнопки

```
Готовий MCP-сервер: ЗНАЙДЕНО, офіційний
Посилання: https://calendarmcp.googleapis.com/mcp/v1
           інструкція: https://developers.google.com/workspace/calendar/api/guides/configure-mcp-server
           (є і в каталозі Connectors у Claude; альтернатива від спільноти —
           https://github.com/taylorwilsdon/google_workspace_mcp)
Автор: Google — офіційна компанія-власник сервісу (статус Developer Preview)
Тип сервера: віддалений
Які інструменти сервера мені потрібні: create_event; корисні також list_events
   і search_events — щоб не створити дубль події, яка вже стоїть у календарі
Чи є серед них дії на запис, видалення, надсилання: ТАК — create_event, delete_event,
   respond_to_event. Це найнебезпечніший сервер із п'яти
Які права/доступи потрібні: OAuth зі скоупом на календар. У мене вже є refresh token
   на ОДИН календар — сервер попросить ширший доступ, до всіх календарів акаунта
Мій висновок: підключу — але з тією самою умовою, що діє в агенті зараз: інструмент
   викликає КОД у callback_run після натискання кнопки власником, а не модель під
   час читання листів. Автопідключення в стилі Кроку 3 (tools=[mcp_client.session])
   тут заборонене: DeepWiki лише читає, а create_event пише в мій календар, і одного
   рядка «Додай подію…» у тілі листа вистачило б, щоб модель це зробила.
   Поки офіційний сервер у Developer Preview, робочий варіант — власний виклик
   Calendar API, а сервер узяти, коли він вийде зі статусу preview.
```

---

### Підсумок для уроку 7

Із п'яти інструментів готовий сервер виправданий лише для одного (`create_calendar_event`), і то з обмеженням на спосіб виклику. Решта чотири — кандидати у **власний MCP-сервер**: пошта з курсором UID, Telegram із кнопками і стан раннера. Саме їх беру на наступне заняття.

---

# Бонус. Той самий агент із **локальним** сервером

## Б.1. Файл `weather_server.py`

In [ ]:
%%writefile weather_server.py
from fastmcp import FastMCP

mcp = FastMCP("Погода")

@mcp.tool
def get_weather(city: str) -> str:
    """Поточна погода в місті. Використовуй, коли питають про погоду."""
    return f"{city}: +7°C, хмарно"

if __name__ == "__main__":
    mcp.run()

## Б.2. Перевірка сервера з терміналу — та сама команда, що в Кроці 1

In [6]:
!fastmcp list weather_server.py --input-schema

Tools (1)

  get_weather(city: str) -> dict
    Поточна погода в місті. Використовуй, коли питають про погоду.
    Input: {"type": "object", "additionalProperties": false, "properties": 
{"city": {"type": "string"}}, "required": ["city"]}

## Б.3–Б.4. Агент із локальним сервером

У коді агента змінено рівно **два** рядки: адреса сервера → ім'я файлу, і питання.

In [7]:
import asyncio
from pathlib import Path
from fastmcp import Client
from fastmcp.client.transports import PythonStdioTransport
from google import genai
from google.genai import types, errors

MODEL = "gemini-3.7-flash"
QUESTION = "Яка зараз погода в Києві?"          # ← зміна 1

# ← зміна 2: замість адреси сервера — локальний файл.
#
# Поза Colab вистачило б одного рядка: Client("weather_server.py").
# Але в Colab stdio-транспорт віддає дочірньому процесу sys.stderr ноутбука,
# а ipykernel підмінив його об'єктом без fileno() — і клієнт падає з
# "RuntimeError: Client failed to connect: fileno".
# log_file=Path(...) підсовує транспорту справжній файл під stderr сервера.
mcp_client = Client(
    PythonStdioTransport(
        script_path=Path("weather_server.py"),
        log_file=Path("weather_server.log"),
    )
)
gemini = genai.Client()


async def ask_once():
    async with mcp_client:
        config = types.GenerateContentConfig(temperature=0, tools=[mcp_client.session])
        return await gemini.aio.models.generate_content(
            model=MODEL, contents=QUESTION, config=config
        )


async def main():
    for attempt in range(4):                     # пауза — поза сесією і через await
        try:
            response = await ask_once()
            break
        except (errors.ClientError, errors.ServerError) as e:
            if e.code in (429, 503) and attempt < 3:
                pause = 20 * (attempt + 1)
                print(f"{e.code}: чекаю {pause} с і повторюю")
                await asyncio.sleep(pause)
                continue
            raise
        except Exception as e:                   # мережа могла впасти і на підключенні до сервера
            if attempt < 3:
                pause = 20 * (attempt + 1)
                print(f"{type(e).__name__}: чекаю {pause} с і повторюю")
                await asyncio.sleep(pause)
                continue
            raise

    print(response.text)

    print("\n--- Що відбулося всередині ---")
    for content in response.automatic_function_calling_history or []:
        for part in content.parts or []:
            if part.function_call:
                print("інструмент:", part.function_call.name)
                print("аргументи:", dict(part.function_call.args))


await main()

Зараз у Києві близько **+7°C**, хмарно.

--- Що відбулося всередині ---
інструмент: get_weather
аргументи: {'city': 'Kyiv'}

### Відповіді на питання Б.4

```
Що змінилось у коді агента, коли сервер став локальним?
   По суті — один аргумент: Client("https://mcp.deepwiki.com/mcp") → локальний файл
   сервера. Більше нічого: ні async with, ні tools=[mcp_client.session], ні розбір
   automatic_function_calling_history не змінились. FastMCP сам зрозумів, що .py —
   це stdio-транспорт: він запустив сервер окремим процесом і говорить із ним через
   стандартний ввід-вивід замість HTTP. Для моделі різниці немає взагалі — вона
   бачить той самий список інструментів зі схемами.

   Одна поправка на середовище: поза ноутбуком достатньо Client("weather_server.py"),
   а в Colab довелося скласти транспорт явно, з log_file. Причина не в MCP: stdio
   віддає дочірньому процесу sys.stderr, а ipykernel підмінив його об'єктом без
   fileno(), тож підключення падає з "Client failed to connect: fileno". log_file
   підсовує справжній файл — і все стартує. Побічний бонус: помилки самого сервера
   тепер лежать у weather_server.log, а не губляться у виводі клітинки.
   Практична різниця не в коді, а навколо: локальний сервер не ходить у мережу,
   не має авторизації, живе рівно стільки, скільки агент, і його схема залежить
   від мого ж файлу — а не від чужого сервісу, який може змінитись без попередження.

Чим схема з fastmcp list схожа на опис get_weather, який ви писали руками на уроці 5?
   Це той самий опис, просто зібраний автоматично:
      name        ← ім'я функції Python                     (get_weather)
      description ← рядок документації функції              ("Поточна погода в місті…")
      parameters  ← анотації типів                          (city: str → {"type": "string"})
      required    ← параметри без значення за замовчуванням  (["city"])
   На уроці 5 я писав(ла) цей словник руками і мусив(ла) стежити, щоб він не
   розійшовся з кодом функції. Тут @mcp.tool будує його з самої функції, тому
   розійтися вони не можуть. Що НЕ будується автоматично — якість тексту:
   формулювання "Використовуй, коли питають про погоду" я все одно пишу сам(а)
   в docstring, бо саме за ним модель вирішує, викликати інструмент чи ні.

Дрібниця, яку видно у виводі: модель передала city='Kyiv' латиницею, хоча питання
   було українською. Схема цього не забороняє — у ній просто "type": "string".
   Якби мій сервер очікував назву українською, треба було б або enum, або пряма
   вказівка в описі параметра. Той самий урок, що і в Кроці 3: аргументи модель
   формулює по-своєму, поки схема не каже інакше.
```

---

# Чек-лист перед здачею

- ☑ Крок 1: вивід `fastmcp list`, усі 3 інструменти з поясненнями, розбір схеми `ask_question` за шаблоном + порівняння з `get_weather`
- ☑ Крок 2: дві команди `fastmcp call` для власного репозиторію (`python-telegram-bot/python-telegram-bot`, не `modelcontextprotocol/python-sdk`) + результати + відповідь 2.4
- ☑ Крок 3: прогноз записано до запуску (клітинка стоїть перед клітинкою запуску)
- ☑ Крок 3: повний вивід програми з блоком «Що відбулося всередині» — два виклики `ask_question` з аргументами
- ☑ Крок 3: порівняння прогнозу з реальністю (інструмент збігся, аргументи частково, кількість викликів — ні)
- ☑ Крок 4: 5 карток з автором, правами і висновком
- ☑ Бонус: локальний `weather_server.py`, його схема і прогін агента
- ☑ Ключа курсу немає ні в коді, ні у виводі: він читається з секретів Colab у змінну середовища
- ☐ Перейменувати файл на `ДЗ6_Прізвище_Імʼя` перед здачею